## Held-out 评测 v1：底座 vs v15（DPO 后）

- **目的**：自测 15 题已不能当评判标准——训练数据就是照着它们的毛病建的。本评测用 25 道从未进过训练的新题（见 `data/heldout-eval-v1.jsonl`），量 v15 的真实水平，并与底座在同一 kernel、同一解码条件下对比。
- **做法**：先跑底座（Qwen2.5-7B-Instruct，未微调），再加载 v15 的 DPO 适配器跑一遍；贪心解码（do_sample=False，max_new_tokens 320），保证可复现；ChatML 与 system 提示与自测相同。
- **评分**：kernel 内自动检查（must_contain 全出现、must_not_contain 全不出现）；must_contain 为空的题只拦禁用串，最终结论由核查人工读答案判定。
- **预期**：约 10–15 分钟。结果存 `outputs/heldout-eval-v1-results.json`。


In [ ]:
# 1. 安装（Kaggle Notebook 里跑一次即可）
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q trl peft accelerate bitsandbytes datasets


In [ ]:
# 2. 配置与评测题（题目硬编码在笔记本里，不走数据集挂载，避开路径问题）
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"  # 4-bit 底座
MAX_SEQ_LENGTH = 2048
MAX_NEW_TOKENS = 320
SYS = "你是一位精通中国历代官制的学者，回答要准确、克制，有出处意识，不确定的地方明说不确定。"
# v15 的 DPO 适配器：训练 kernel 最新成功版（v15）的输出，以 kernel_sources 挂载。
# 顶层目录本身就是合法的适配器目录（含 adapter_model.safetensors 与 adapter_config.json）。
ADAPTER_CANDIDATES = [
    "/kaggle/input/officialdom-qwen2-5-7b-sft-batch-1/outputs/officialdom-qwen2.5-7b-sft-v6-dpo",
    "/kaggle/input/officialdom-qwen2-5-7b-sft-batch-1/officialdom-qwen2.5-7b-sft-v6-dpo",
]
EVAL = [{"category": "fiscal", "expected": "2025年新增政府债务总规模11.86万亿元，比上年增加1.6万亿元。", "id": "A1", "must_contain": ["11.86", "1.6万亿"], "must_not_contain": ["5.89"], "notes": "2025年报告原文核对：新增政府债务总规模11.86万亿元，比上年增加1.6万亿元。5.89万亿是2026年赤字规模，混淆即错。", "question": "2025年新增政府债务总规模是多少？比上年增加多少？"}, {"category": "fiscal", "expected": "2026年经济增长目标是4.5%－5%。", "id": "A2", "must_contain": ["4.5%"], "must_not_contain": ["5%左右"], "notes": "2026年报告第44行原文：经济增长4.5%－5%。'5%左右'是2025年的目标，答它即把2025年当2026年。", "question": "2026年政府工作报告提出的经济增长目标是多少？"}, {"category": "fiscal", "expected": "三个数字都对：2026年赤字率4%左右，赤字规模5.89万亿元，比上年增加2300亿元。", "id": "A4", "must_contain": ["5.89", "2300亿"], "must_not_contain": ["5.66"], "notes": "2026年报告原文核对。5.66万亿是2025年赤字规模，若出现在答案里且当作2026年的即混淆。结论'都对'需人工核对（自动检查只核数字重述）。", "question": "2026年赤字率4%左右、赤字规模5.89万亿、比上年增加2300亿元，这三个数字都对吗？"}, {"category": "fiscal", "expected": "部署部分的提法是'继续实施更加积极的财政政策'。", "id": "A5", "must_contain": ["继续实施更加积极的财政政策"], "must_not_contain": ["实施更加积极的财政政策和适度宽松的货币政策"], "notes": "2026年报告核对：'继续实施更加积极的财政政策'是部署部分；'实施更加积极的财政政策和适度宽松的货币政策'是回顾部分述2025年工作的合并句，引用它即答错部分。", "question": "2026年政府工作报告部署部分，财政政策的提法是什么？注意不要引用回顾部分的句子。"}, {"category": "fiscal", "expected": "对。2025年赤字规模5.66万亿元，2024年4.06万亿元，相差1.6万亿元。", "id": "A6", "must_contain": ["5.66", "4.06", "1.6"], "must_not_contain": ["11.86"], "notes": "5.66-4.06=1.60，数字上说法成立。注意1.6万亿同时也是2025年新增政府债务的增加额，若答案引用11.86万亿即混淆了两个1.6万亿。结论'对'需人工核对。", "question": "有人说2025年赤字规模比2024年多1.6万亿，这个说法对吗？请用两年的赤字规模数字说明。"}, {"category": "cbdb", "expected": "1148年。", "id": "B1", "must_contain": ["1148"], "must_not_contain": ["1478"], "notes": "CBDB亲核：王佐 personid 13477，生1129年、卒1191年，ENTRY_DATA entry code 36，c_year 1148。1478年是另一位王佐（1440–1512）的进士年份。", "question": "生于1129年、卒于1191年的王佐，是哪一年进士？"}, {"category": "cbdb", "expected": "1478年。", "id": "B2", "must_contain": ["1478"], "must_not_contain": ["1148"], "notes": "CBDB亲核：王佐 personid 67292，生1440年、卒1512年，entry code 36，c_year 1478。", "question": "生于1440年、卒于1512年的王佐，是哪一年进士？"}, {"category": "cbdb", "expected": "生于1129年的王佐先中进士（1148年），比生于1440年的王佐（1478年）早330年。", "id": "B3", "must_contain": ["1148", "330"], "must_not_contain": [], "notes": "1478-1148=330。CBDB亲核（同B1/B2）。", "question": "生于1129年的王佐和生于1440年的王佐，谁先中进士？早多少年？"}, {"category": "cbdb", "expected": "1442年。", "id": "B4", "must_contain": ["1442"], "must_not_contain": [], "notes": "CBDB亲核：李實 personid 67447，生1413年、卒1485年，entry code 36，c_year 1442。", "question": "生于1413年、卒于1485年的李實，是哪一年进士？"}, {"category": "cbdb", "expected": "1505年。", "id": "B5", "must_contain": ["1505"], "must_not_contain": ["1478"], "notes": "CBDB亲核：張經 personid 201603，生1474年，entry code 36，c_year 1505。1478年是另一位張經（生1452年）的进士年份。", "question": "生于1474年的張經，是哪一年进士？"}, {"category": "regulation", "expected": "按清代规矩：勘灾、报府、请赈、开仓放赈、造册核销。不得引用任何现代法规（清代无现代法规）。", "id": "C1", "must_contain": [], "must_not_contain": ["《中国共产党问责条例》", "《党政机关厉行节约反对浪费条例》", "《信访工作条例》", "《中华人民共和国防洪法》"], "notes": "时代匹配题：清代场景，现代法规名一出现即错。程序细节人工核对。", "question": "清代县令遇到水灾，需要开仓赈灾，按清代规矩应当如何处理？"}, {"category": "regulation", "expected": "《中国共产党问责条例》第七条第九项：履行管理、监督职责不力，职责范围内发生重特大生产安全事故、公共安全事件。", "id": "C2", "must_contain": ["第七条", "第九项"], "must_not_contain": ["第三项"], "notes": "问责条例第七条第九项原文已核对（党内法规原文），覆盖防汛失职与生产安全事故。第七条第三项是Q9在v10的旧错法。", "question": "某县发生重特大生产安全事故，要追究领导责任，依据《中国共产党问责条例》的哪一条？"}, {"category": "regulation", "expected": "不能。父子、兄弟等亲属不得在同一衙门或同一地方任职（亲属回避），官员也不得在本籍任职（本籍回避）。", "id": "C3", "must_contain": ["回避"], "must_not_contain": ["《公务员回避规定》", "第十九条"], "notes": "时代匹配题：清代回避靠清代规矩，现代《公务员回避规定》第十九条是v11 Q15的旧错法。", "question": "清代知县的儿子考中进士后，能不能在父亲任职的县里做官？按什么规矩？"}, {"category": "regulation", "expected": "《中华人民共和国公务员法》不适用于清代。清代考核是京察（京官）与大计（外官），三年一次。", "id": "C4", "must_contain": ["京察", "大计"], "must_not_contain": ["平时考核"], "notes": "陷阱题（同Q15结构，换陷阱）：现代法规名可以出现（为反驳它），但若答案用现代平时考核办法作答即时代错置。'公务员法不适用清代'需人工核对。", "question": "按《中华人民共和国公务员法》，清代知县的考核应当怎么进行？"}, {"category": "regulation", "expected": "检验尸伤、取供、详报知府与按察司，死刑须上报复核（秋审/朝审）；知县无权擅自处决。", "id": "C6", "must_contain": [], "must_not_contain": ["《刑事诉讼法》", "《公安机关办理刑事案件程序规定》"], "notes": "时代匹配题：清代命案程序，现代刑诉法名一出现即错。程序细节人工核对。", "question": "清代知县审理命案，必须经过哪些程序？能不能直接判决？"}, {"category": "institution", "expected": "军机处是皇帝的秘书决策班子，无正式衙署编制但实权大；内阁是正式机构，到清代实权已转移到军机处。", "id": "D1", "must_contain": ["军机处", "内阁"], "must_not_contain": [], "notes": "与Q1同主题、不同问法（比较而非定义）。细节人工核对。", "question": "清代军机处和内阁在权力上有什么区别？"}, {"category": "institution", "expected": "康熙朝创立，雍正朝推广、制度化。有资格者为亲王、大学士、总督、巡抚等高级官员及皇帝特许者。", "id": "D2", "must_contain": ["康熙"], "must_not_contain": [], "notes": "标准说法为康熙创立、雍正制度化；若只答'雍正创立'算部分正确，人工核对时扣分。", "question": "清代哪些官员有资格上密折？密折制度是谁创立的？"}, {"category": "institution", "expected": "1853年为镇压太平天国而设，在货物通过厘卡时抽税（值百抽一，约1%），主要用于军饷，后成为地方财政重要来源。", "id": "D4", "must_contain": ["太平天国"], "must_not_contain": [], "notes": "厘金讲座（2026-11-05）主题。税率与用途人工核对。", "question": "清代厘金制度是怎样征收的？主要用来做什么？"}, {"category": "institution", "expected": "院试考中为秀才（生员），乡试考中为举人，会试考中为贡士，殿试考中为进士。", "id": "D5", "must_contain": ["秀才", "举人", "进士"], "must_not_contain": [], "notes": "科举分级常识。", "question": "清代科举考试分哪几级？考中后分别获得什么身份？"}, {"category": "institution", "expected": "不能直接当作历史事实。小说是文学创作，反映的是民间观念；须对照法规、史料、档案核对（训练方案的三层三角校准）。", "id": "D6", "must_contain": ["不能"], "must_not_contain": [], "notes": "方法题：小说作标注的民间观念，不是史实本身。'不能'需人工核对语境（须是否定'直接当史实'，而非否定小说的价值）。", "question": "小说里写的官场规矩，能直接当作历史事实引用吗？为什么？"}, {"category": "consistency", "expected": "不是同一回事。11.86万亿是2025年新增政府债务总规模，5.89万亿是2026年赤字规模。", "id": "E1", "must_contain": ["11.86", "5.89"], "must_not_contain": [], "notes": "区分测试：模型在Q10曾把11.86万亿塞进赤字率答案。人工核对是否明确区分两者。", "question": "2025年新增政府债务11.86万亿，2026年赤字5.89万亿，这两个数字说的是同一回事吗？"}, {"category": "consistency", "expected": "两年都有。2025年报告与2026年报告都写'赤字率拟按4%左右安排'。", "id": "E2", "must_contain": ["2025", "2026"], "must_not_contain": [], "notes": "两份报告原文均已核对（grep确认'赤字率拟按4%左右安排'在两份文件中都出现）。若只答其中一年即错，人工核对。", "question": "“赤字率拟按4%左右安排”这句话，出现在2025年报告里，还是2026年报告里？"}, {"category": "consistency", "expected": "晚清小说提供民间观念（待验证的说法）；党内法规与政府工作报告是当代材料，不能用来证明清代的做法；证明清代做法须用清代法规、史料、档案。", "id": "E3", "must_contain": [], "must_not_contain": [], "notes": "方法题（训练方案三层三角校准的时代版）。全靠人工核对：是否明确指出当代材料不能证明清代做法。", "question": "研究清代官场规矩时，晚清小说、党内法规、政府工作报告，这三类材料各有什么用？哪一类不能用来证明清代的做法？"}, {"category": "consistency", "expected": "高1个百分点（2025年4%左右，2024年3%）。", "id": "E5", "must_contain": ["1个百分点"], "must_not_contain": [], "notes": "算术题：数字已在题中给出，考计算与不自相矛盾。注意Q14答案曾错误使用'比上年提高1个百分点'。", "question": "2025年政府工作报告说“赤字率拟按4%左右安排”，2024年报告说“赤字率拟按3%安排”，2025年比2024年高几个百分点？"}, {"category": "consistency", "expected": "最大不同：清代知县是父母官、行政与司法合一，按律例、会典断案；现代县长不直接审案，司法由法院负责。时代与制度都不同。", "id": "E6", "must_contain": [], "must_not_contain": [], "notes": "时代比较题。全靠人工核对：是否区分清代（行政司法合一）与现代（司法独立于行政），以及是否把现代法规当作清代依据。", "question": "清代知县和现代县长，在处理百姓告状这件事上最大的不同是什么？"}]
print("评测题：", len(EVAL), "道")
from collections import Counter as _Counter
print("分类：", dict(_Counter(r["category"] for r in EVAL)))


In [ ]:
# 3. 定位 v15 的 DPO 适配器（候选路径 + 全盘搜索；找不到就大声报错）
import json, os

def _list_input(root="/kaggle/input"):
    for dirpath, dirnames, filenames in os.walk(root):
        level = dirpath.replace(root, "").count(os.sep)
        print("  " * level + os.path.basename(dirpath) + "/")
        for f in filenames:
            print("  " * (level + 1) + f)

print("=== /kaggle/input 实际挂载内容 ===")
_list_input()
print("=== DPO 适配器路径检查 ===")
ADAPTER_PATH = None
for p in ADAPTER_CANDIDATES:
    print(("OK  " if os.path.exists(os.path.join(p, "adapter_model.safetensors")) else "缺失") + " " + p)
    if os.path.exists(os.path.join(p, "adapter_model.safetensors")):
        ADAPTER_PATH = p
        break
if ADAPTER_PATH is None:
    print("=== 候选路径全缺失，全盘搜索 DPO 适配器 ===")
    for dirpath, dirnames, filenames in os.walk("/kaggle/input"):
        if "adapter_model.safetensors" in filenames and "ref" not in dirpath.split(os.sep):
            if "dpo" in dirpath:
                ADAPTER_PATH = dirpath
                print("搜到：", ADAPTER_PATH)
                break
if ADAPTER_PATH is None:
    raise FileNotFoundError("v15 的 DPO 适配器不在任何候选路径，见上方挂载清单。kernel_sources 是否挂了训练 kernel 的最新成功版输出？")
print("使用适配器：", ADAPTER_PATH)
print("适配器配置：", json.load(open(os.path.join(ADAPTER_PATH, "adapter_config.json"))))


In [ ]:
# 4. 底座评测：加载底座（未微调），贪心跑 25 题
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

def _run_eval(tag):
    answers = {}
    for r in EVAL:
        prompt = (f"<|im_start|>system\n{SYS}<|im_end|>\n"
                  f"<|im_start|>user\n{r['question']}<|im_end|>\n<|im_start|>assistant\n")
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, use_cache=True,
                             do_sample=False)
        gen_ids = out[0][inputs["input_ids"].shape[1]:]
        ans = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
        answers[r["id"]] = ans
        print(f"\n===== [{tag}] {r['id']} =====\n问：{r['question']}\n答：{ans}\n")
        print(f"（生成 token 数：{len(gen_ids)}，答案字数：{len(ans)}）")
    torch.cuda.empty_cache()
    return answers

print("########## 底座开始评测 ##########")
base_answers = _run_eval("底座")
print("底座评测完成。")
del model
torch.cuda.empty_cache()
print("底座模型已释放。")


In [ ]:
# 5. v15（DPO 后）评测：加载 DPO 适配器（与 v15 第 4 格相同做法），贪心跑 25 题
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("DPO 适配器加载完成（可训练参数：", _trainable, "；评测只用推理，不训练）")
FastLanguageModel.for_inference(model)

print("########## v15（DPO 后）开始评测 ##########")
dpo_answers = _run_eval("v15-DPO")
print("v15 评测完成。")


In [ ]:
# 6. 自动打分表 + 保存结果
import json as _json

def _auto(ans, r):
    ok_must = all(s in ans for s in r["must_contain"])
    ok_not = all(s not in ans for s in r["must_not_contain"])
    manual = (len(r["must_contain"]) == 0 and len(r["must_not_contain"]) == 0)
    return ok_must and ok_not, manual

print("\n===== 自动打分表（✓=自动通过，✗=自动不通过，?=须人工读答案） =====")
print(f"{'题号':<5}{'类别':<12}{'底座':<6}{'v15':<6}")
tot_base = tot_dpo = 0
for r in EVAL:
    b_ok, b_man = _auto(base_answers[r["id"]], r)
    d_ok, d_man = _auto(dpo_answers[r["id"]], r)
    b_s = "?" if b_man else ("✓" if b_ok else "✗")
    d_s = "?" if d_man else ("✓" if d_ok else "✗")
    if b_ok and not b_man:
        tot_base += 1
    if d_ok and not d_man:
        tot_dpo += 1
    print(f"{r['id']:<5}{r['category']:<12}{b_s:<6}{d_s:<6}  {r['question'][:40]}")
print(f"\n自动通过（不含须人工的题）：底座 {tot_base}/25，v15 {tot_dpo}/25")
print("注意：'?' 的题（must_contain 与 must_not_contain 均为空）自动检查不判分，须人工读答案；'✓' 也不等于全对，只表示必含的串都出现、禁用的串都没出现，最终结论以人工评分为准。")

os.makedirs("outputs", exist_ok=True)
results = {
    "eval_version": "heldout-v1",
    "questions": EVAL,
    "base_answers": base_answers,
    "dpo_answers": dpo_answers,
    "auto_score": {"base": tot_base, "dpo": tot_dpo, "total": len(EVAL)},
}
with open("outputs/heldout-eval-v1-results.json", "w", encoding="utf-8") as f:
    _json.dump(results, f, ensure_ascii=False, indent=2)
print("\n结果已存到 outputs/heldout-eval-v1-results.json")
print("=== outputs/ 全部文件 ===")
for _root, _dirs, _files in os.walk("outputs"):
    for _f in _files:
        print(os.path.join(_root, _f))
print("完成。")
